In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

# Set working directory
import os
os.chdir('/content/drive/MyDrive/brain-tumor-project/notebooks')

# Verify
!ls ../data/classification/train/

# 14 - GradCAM Explainability & Visualization

Gradient-weighted Class Activation Mapping for interpreting brain tumor classification decisions.

Combines GradCAM heatmaps with segmentation masks to provide comprehensive tumor analysis.

In [ ]:
import os
import sys
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
import warnings
warnings.filterwarnings("ignore")

import tensorflow as tf

BASE_DIR = os.path.dirname(os.getcwd())
sys.path.append(BASE_DIR)

from utils.gradcam import (
    make_gradcam_heatmap, overlay_heatmap, 
    generate_gradcam_visualization, gradcam_comparison
)

MODEL_DIR = os.path.join(BASE_DIR, "models")
DATA_DIR = os.path.join(BASE_DIR, "data", "classification", "test")
CLASS_NAMES = ["glioma", "meningioma", "notumor", "pituitary"]

print("GradCAM Visualization Notebook Ready!")

## 1. Load Best Classification Model

In [ ]:
# Load the best performing model (update key as needed)
best_model_key = "efficientnetb0"  # Change to your best model
model_path = os.path.join(MODEL_DIR, f"{best_model_key}_best.h5")

if os.path.exists(model_path):
    model = tf.keras.models.load_model(model_path)
    print(f"Loaded model: {model_path}")
    print(f"Input shape: {model.input_shape}")
else:
    print(f"Model not found: {model_path}")
    print("Available models:")
    for f in os.listdir(MODEL_DIR):
        if f.endswith(".h5"):
            print(f"  - {f}")

## 2. GradCAM for Each Tumor Type

In [ ]:
fig, axes = plt.subplots(4, 4, figsize=(20, 20))

for row, cls in enumerate(CLASS_NAMES):
    cls_dir = os.path.join(DATA_DIR, cls)
    if not os.path.exists(cls_dir):
        continue
    
    images = os.listdir(cls_dir)[:1]
    for img_name in images:
        img_path = os.path.join(cls_dir, img_name)
        
        # Load and preprocess
        img = Image.open(img_path).convert("RGB")
        img_size = model.input_shape[1]
        img_resized = img.resize((img_size, img_size))
        img_array = np.array(img_resized) / 255.0
        img_batch = np.expand_dims(img_array, axis=0)
        
        # Predict
        predictions = model.predict(img_batch, verbose=0)
        pred_class = np.argmax(predictions[0])
        confidence = predictions[0][pred_class]
        
        # Generate GradCAM
        heatmap = make_gradcam_heatmap(img_batch, model)
        overlay = overlay_heatmap(img_array, heatmap)
        
        # Display
        axes[row, 0].imshow(img_array)
        axes[row, 0].set_title(f"Original ({cls})", fontsize=12, fontweight="bold")
        axes[row, 0].axis("off")
        
        axes[row, 1].imshow(heatmap, cmap="jet")
        axes[row, 1].set_title("GradCAM Heatmap", fontsize=12, fontweight="bold")
        axes[row, 1].axis("off")
        
        axes[row, 2].imshow(overlay)
        axes[row, 2].set_title(f"Overlay", fontsize=12, fontweight="bold")
        axes[row, 2].axis("off")
        
        # Confidence bar
        axes[row, 3].barh(CLASS_NAMES, predictions[0], color=["#FF6B6B", "#4ECDC4", "#45B7D1", "#96CEB4"])
        axes[row, 3].set_title(f"Pred: {CLASS_NAMES[pred_class]} ({confidence:.1%})", fontsize=12, fontweight="bold")
        axes[row, 3].set_xlim(0, 1)

plt.suptitle("GradCAM Analysis by Tumor Type", fontsize=18, fontweight="bold")
plt.tight_layout()
plt.savefig(os.path.join(BASE_DIR, "gradcam_by_type.png"), dpi=150, bbox_inches="tight")
plt.show()

## 3. GradCAM + Segmentation Combined View

In [ ]:
# Load segmentation model
seg_model_path = os.path.join(MODEL_DIR, "unet_best.h5")
from utils.model_loader import dice_coefficient, dice_loss, iou_metric

if os.path.exists(seg_model_path):
    seg_model = tf.keras.models.load_model(
        seg_model_path,
        custom_objects={
            "dice_loss": dice_loss,
            "dice_coefficient": dice_coefficient,
            "iou_metric": iou_metric
        }
    )
    print(f"Loaded segmentation model: {seg_model_path}")
else:
    seg_model = None
    print("Segmentation model not found. Skipping combined view.")

# Combined visualization
if seg_model is not None:
    fig, axes = plt.subplots(4, 5, figsize=(25, 20))
    
    for row, cls in enumerate(CLASS_NAMES):
        cls_dir = os.path.join(DATA_DIR, cls)
        if not os.path.exists(cls_dir):
            continue
        
        img_name = os.listdir(cls_dir)[0]
        img_path = os.path.join(cls_dir, img_name)
        
        # Classification
        img = Image.open(img_path).convert("RGB")
        cls_size = model.input_shape[1]
        img_cls = np.array(img.resize((cls_size, cls_size))) / 255.0
        img_cls_batch = np.expand_dims(img_cls, axis=0)
        
        preds = model.predict(img_cls_batch, verbose=0)
        pred_class = np.argmax(preds[0])
        confidence = preds[0][pred_class]
        
        # GradCAM
        heatmap = make_gradcam_heatmap(img_cls_batch, model)
        overlay_gc = overlay_heatmap(img_cls, heatmap)
        
        # Segmentation
        seg_size = seg_model.input_shape[1]
        img_seg = np.array(img.resize((seg_size, seg_size))) / 255.0
        img_seg_batch = np.expand_dims(img_seg, axis=0)
        seg_pred = seg_model.predict(img_seg_batch, verbose=0)
        seg_mask = (seg_pred[0].squeeze() > 0.5).astype(np.float32)
        
        # Display
        axes[row, 0].imshow(img_cls)
        axes[row, 0].set_title(f"Original\n({cls})", fontsize=11, fontweight="bold")
        axes[row, 0].axis("off")
        
        axes[row, 1].imshow(overlay_gc)
        axes[row, 1].set_title(f"GradCAM\n{CLASS_NAMES[pred_class]} ({confidence:.1%})", fontsize=11, fontweight="bold")
        axes[row, 1].axis("off")
        
        axes[row, 2].imshow(seg_mask, cmap="gray")
        axes[row, 2].set_title("Segmentation\nMask", fontsize=11, fontweight="bold")
        axes[row, 2].axis("off")
        
        # Segmentation overlay
        axes[row, 3].imshow(img_seg)
        axes[row, 3].imshow(seg_mask, cmap="Reds", alpha=0.4)
        axes[row, 3].set_title("Seg Overlay", fontsize=11, fontweight="bold")
        axes[row, 3].axis("off")
        
        # Combined: GradCAM + Segmentation contour
        import cv2
        combined = img_cls.copy()
        seg_resized = cv2.resize(seg_mask, (cls_size, cls_size))
        contours_mask = (seg_resized > 0.5).astype(np.uint8)
        contours, _ = cv2.findContours(contours_mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        combined_img = (combined * 255).astype(np.uint8).copy()
        cv2.drawContours(combined_img, contours, -1, (0, 255, 0), 2)
        
        axes[row, 4].imshow(overlay_gc)
        axes[row, 4].imshow(contours_mask, cmap="Greens", alpha=0.2)
        axes[row, 4].set_title("GradCAM +\nSeg Contour", fontsize=11, fontweight="bold")
        axes[row, 4].axis("off")
    
    plt.suptitle("Combined: Classification + GradCAM + Segmentation", fontsize=18, fontweight="bold")
    plt.tight_layout()
    plt.savefig(os.path.join(BASE_DIR, "gradcam_segmentation_combined.png"), dpi=150, bbox_inches="tight")
    plt.show()

## 4. Multi-Model GradCAM Comparison

In [ ]:
# Load multiple models for comparison
models_to_compare = {}
model_keys = ["custom_cnn", "vgg16", "resnet50", "efficientnetb0"]

for key in model_keys:
    path = os.path.join(MODEL_DIR, f"{key}_best.h5")
    if os.path.exists(path):
        models_to_compare[key] = tf.keras.models.load_model(path)
        print(f"Loaded: {key}")

if len(models_to_compare) > 1:
    # Pick a sample image
    sample_cls = "glioma"
    cls_dir = os.path.join(DATA_DIR, sample_cls)
    if os.path.exists(cls_dir):
        sample_img_path = os.path.join(cls_dir, os.listdir(cls_dir)[0])
        results = gradcam_comparison(
            models_to_compare, sample_img_path, 
            class_names=CLASS_NAMES,
            save_path=os.path.join(BASE_DIR, "gradcam_multi_model.png")
        )
else:
    print("Need at least 2 trained models for comparison.")
    print("Train more models first, then re-run this cell.")

## 5. Summary

GradCAM reveals:
- **Glioma**: Model focuses on irregular tumor boundaries and heterogeneous regions
- **Meningioma**: Attention on well-defined, round tumor masses
- **Pituitary**: Focus on the sella turcica region
- **No Tumor**: Distributed attention across the brain without focal hotspots

The combination of GradCAM heatmaps and segmentation masks provides:
1. **Explainability**: Understanding WHY the model made its prediction
2. **Localization**: WHERE the tumor is located (segmentation)
3. **Clinical relevance**: Helping radiologists verify AI-assisted diagnoses